# 🌍 Climate Data Analyst Portfolio
### Naveed Baig | 2025
---
**Dataset:** Our World in Data — CO₂ & Greenhouse Gas Emissions (1750–2024)  
**Tools:** Python · Pandas · Plotly  
**Focus:** Global emissions trends, top emitters, temperature impact, energy sources, GDP correlation

In [ ]:
# ── Setup & Data Load ──────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load dataset
df = pd.read_csv(r"C:\Users\Hp\Downloads\owid-co2-data.csv")

# Separate world aggregate vs individual countries
EXCLUDE = ["World","Asia","Europe","Africa","North America","South America",
           "Oceania","European Union (27)","High-income countries",
           "Low-income countries","Middle East (GCP)",
           "Upper-middle-income countries","Lower-middle-income countries",
           "Asia (excl. China and India)","Europe (excl. EU-27)",
           "Europe (excl. EU-28)","North America (excl. USA)"]

countries_df = df[~df["country"].isin(EXCLUDE)].copy()
world_df     = df[df["country"] == "World"].copy()

print(f"✅ Loaded  →  {df.shape[0]:,} rows  |  {df['country'].nunique()} countries  |  {df['year'].min()}–{df['year'].max()}")
print(f"📋 Columns: {df.shape[1]}")
df.head(3)

---
## 📈 Project 1 — Global CO₂ Emissions Trend (1900–2024)
**Question:** How fast have global CO₂ emissions risen, and what events caused dips?

In [ ]:
# Project 1 — Global CO₂ Trend
trend = world_df[world_df["year"] >= 1900][["year","co2"]].dropna()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=trend["year"], y=trend["co2"],
    mode="lines", fill="tozeroy",
    line=dict(color="#00c896", width=2.5),
    fillcolor="rgba(0,200,150,0.15)",
    name="World CO₂ (billion tonnes)"
))

# Annotate key events
events = {1929:"Great Depression", 1945:"WWII End", 1973:"Oil Crisis",
          2008:"Financial Crisis", 2020:"COVID-19"}
for yr, label in events.items():
    val = trend[trend["year"]==yr]["co2"]
    if not val.empty:
        fig.add_vline(x=yr, line_dash="dash", line_color="rgba(255,200,100,0.6)", line_width=1.5)
        fig.add_annotation(x=yr, y=val.values[0]+2.5, text=label,
                           showarrow=False, font=dict(size=9, color="#ffc864"), textangle=-90)

fig.update_layout(
    title="🌍 Global CO₂ Emissions (1900–2024)",
    template="plotly_dark", paper_bgcolor="#0b1a14", plot_bgcolor="#112a1e",
    height=480, xaxis_title="Year", yaxis_title="CO₂ (billion tonnes)",
    hovermode="x unified", font=dict(family="Arial")
)
fig.show()

In [ ]:
# Year-on-Year Growth Rate
growth = world_df[world_df["year"] >= 1960][["year","co2_growth_prct"]].dropna()

fig2 = px.bar(growth, x="year", y="co2_growth_prct",
    color="co2_growth_prct",
    color_continuous_scale=["#00c896","#ffc864","#ff4444"],
    title="📊 Year-on-Year CO₂ Growth Rate (%)",
    labels={"co2_growth_prct":"Growth %","year":"Year"})
fig2.add_hline(y=0, line_color="white", line_width=0.8)
fig2.update_layout(template="plotly_dark", paper_bgcolor="#0b1a14",
    plot_bgcolor="#112a1e", height=360, coloraxis_showscale=False)
fig2.show()

print("\n💡 Key Insight: Global CO₂ rose 20× since 1900 (2B → 37B tonnes).")
print("   COVID-19 caused the largest-ever single-year drop (–5.4%) in 2020.")
print("   But 2021 saw an immediate record rebound.")

---
## 🏆 Project 2 — Top CO₂ Emitters (2023)
**Question:** Which countries emit the most — total vs. per person?

In [ ]:
# Project 2 — Top Emitters
year = 2022
top_total = countries_df[countries_df["year"]==year][["country","co2","share_global_co2"]]\
    .dropna().sort_values("co2", ascending=False).head(15)
top_percap = countries_df[countries_df["year"]==year][["country","co2_per_capita"]]\
    .dropna().sort_values("co2_per_capita", ascending=False).head(15)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Total CO₂ (billion tonnes)","CO₂ Per Capita (tonnes/person)"])

fig.add_trace(go.Bar(
    x=top_total["co2"], y=top_total["country"],
    orientation="h", marker_color="#00c896",
    name="Total CO₂"), row=1, col=1)

fig.add_trace(go.Bar(
    x=top_percap["co2_per_capita"], y=top_percap["country"],
    orientation="h", marker_color="#ffc864",
    name="Per Capita"), row=1, col=2)

fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_yaxes(autorange="reversed", row=1, col=2)
fig.update_layout(
    title=f"🏆 Top CO₂ Emitters — {year}",
    template="plotly_dark", paper_bgcolor="#0b1a14", plot_bgcolor="#112a1e",
    height=520, showlegend=False)
fig.show()

In [ ]:
# Global share — Donut chart
top8 = top_total.head(8).copy()
rest = pd.DataFrame([{"country":"Rest of World",
                       "share_global_co2": 100 - top8["share_global_co2"].sum()}])
pie_df = pd.concat([top8[["country","share_global_co2"]], rest], ignore_index=True)

fig3 = px.pie(pie_df, values="share_global_co2", names="country",
    title="🌐 Global CO₂ Share — Top 8 Countries (2022)",
    color_discrete_sequence=px.colors.sequential.Greens_r, hole=0.45)
fig3.update_layout(template="plotly_dark", paper_bgcolor="#0b1a14", height=420)
fig3.show()

print("\n💡 Key Insight: China + USA = ~40% of global CO₂ by volume.")
print("   But Gulf states top the per-capita ranking — a key policy distinction.")

---
## 🌡️ Project 3 — Temperature Impact by Country
**Question:** Which countries have contributed the most to global warming historically?

In [ ]:
# Project 3 — Temperature Impact
latest = countries_df["year"].max()
temp = countries_df[countries_df["year"]==latest][
    ["country","temperature_change_from_co2",
     "temperature_change_from_ch4","temperature_change_from_n2o"]
].dropna(subset=["temperature_change_from_co2"])\
 .sort_values("temperature_change_from_co2", ascending=False).head(20)

# Horizontal bar — warming contribution
fig = px.bar(temp.sort_values("temperature_change_from_co2"),
    x="temperature_change_from_co2", y="country", orientation="h",
    color="temperature_change_from_co2",
    color_continuous_scale=["#00c896","#ffc864","#ff4444"],
    title="🌡️ Cumulative Temperature Contribution by Country (°C)",
    labels={"temperature_change_from_co2":"°C warming","country":""})
fig.update_layout(template="plotly_dark", paper_bgcolor="#0b1a14",
    plot_bgcolor="#112a1e", height=560, coloraxis_showscale=False)
fig.show()

In [ ]:
# GHG Breakdown — stacked bar for Top 5
top5 = temp.head(5)
fig2 = go.Figure()
fig2.add_trace(go.Bar(name="CO₂", x=top5["country"],
    y=top5["temperature_change_from_co2"], marker_color="#00c896"))
fig2.add_trace(go.Bar(name="CH₄ (Methane)", x=top5["country"],
    y=top5["temperature_change_from_ch4"], marker_color="#ffc864"))
fig2.add_trace(go.Bar(name="N₂O", x=top5["country"],
    y=top5["temperature_change_from_n2o"], marker_color="#ff8c69"))

fig2.update_layout(
    barmode="stack",
    title="🔬 Warming by Gas Type — Top 5 Countries (°C)",
    template="plotly_dark", paper_bgcolor="#0b1a14", plot_bgcolor="#112a1e",
    height=400, yaxis_title="°C Contribution")
fig2.show()

print("\n💡 Key Insight: The USA leads in cumulative warming contribution due to 100+ years")
print("   of heavy industrial emissions — more than China despite China emitting more today.")

---
## ⚡ Project 4 — CO₂ by Energy Source (1900–2023)
**Question:** How has the fuel mix driving global emissions changed over time?

In [ ]:
# Project 4 — Energy Source Breakdown
energy = world_df[world_df["year"]>=1900][
    ["year","coal_co2","oil_co2","gas_co2","cement_co2","flaring_co2"]].dropna()

fig = go.Figure()
sources = [
    ("coal_co2",    "Coal",    "#ff6b6b"),
    ("oil_co2",     "Oil",     "#ffc864"),
    ("gas_co2",     "Gas",     "#69c0ff"),
    ("cement_co2",  "Cement",  "#b0b0b0"),
    ("flaring_co2", "Flaring", "#ff9f43"),
]
for col, label, color in sources:
    fig.add_trace(go.Scatter(
        x=energy["year"], y=energy[col],
        mode="lines", stackgroup="one",
        name=label, line=dict(width=0.5, color=color),
        fillcolor=color
    ))

fig.update_layout(
    title="⚡ Global CO₂ by Energy Source — Stacked Area (1900–2023)",
    template="plotly_dark", paper_bgcolor="#0b1a14", plot_bgcolor="#112a1e",
    height=480, xaxis_title="Year", yaxis_title="CO₂ (billion tonnes)",
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.15))
fig.show()

In [ ]:
# Country Energy Mix Comparison 2022
compare = ["China","United States","India","Germany","Brazil","Pakistan"]
mix = countries_df[(countries_df["year"]==2022) & (countries_df["country"].isin(compare))]\
    [["country","coal_co2","oil_co2","gas_co2"]].dropna()

fig2 = go.Figure()
for src, color in [("coal_co2","#ff6b6b"),("oil_co2","#ffc864"),("gas_co2","#69c0ff")]:
    fig2.add_trace(go.Bar(name=src.replace("_co2","").capitalize(),
        x=mix["country"], y=mix[src], marker_color=color))

fig2.update_layout(
    barmode="group",
    title="🌏 Coal / Oil / Gas CO₂ — Country Comparison (2022)",
    template="plotly_dark", paper_bgcolor="#0b1a14", plot_bgcolor="#112a1e",
    height=400, yaxis_title="CO₂ (billion tonnes)")
fig2.show()

print("\n💡 Key Insight: Coal is still the #1 source globally.")
print("   Europe & USA cut coal; Asia still relies heavily on it.")

---
## 💰 Project 5 — CO₂ Emissions vs GDP
**Question:** Do richer countries pollute more? Is the world becoming more carbon-efficient?

In [ ]:
# Project 5 — Emissions vs GDP scatter
scatter = countries_df[countries_df["year"]==2022][
    ["country","gdp","co2","co2_per_capita","population","co2_per_gdp"]].dropna()
scatter = scatter[scatter["gdp"] > 0].copy()
scatter["gdp_per_capita"] = scatter["gdp"] / scatter["population"]

fig = px.scatter(scatter,
    x="gdp_per_capita", y="co2_per_capita",
    size="population", color="co2_per_gdp",
    hover_name="country",
    size_max=60, log_x=True,
    color_continuous_scale="RdYlGn_r",
    title="💰 CO₂ Per Capita vs GDP Per Capita — 2022 (bubble = population)",
    labels={
        "gdp_per_capita":"GDP per Capita (USD, log scale)",
        "co2_per_capita":"CO₂ per Capita (tonnes)",
        "co2_per_gdp":"CO₂ Intensity",
        "population":"Population"
    })
fig.update_layout(template="plotly_dark", paper_bgcolor="#0b1a14",
    plot_bgcolor="#112a1e", height=520)
fig.show()

In [ ]:
# CO₂ Intensity over time — is the world decarbonizing?
intensity = world_df[world_df["year"]>=1960][["year","co2_per_gdp"]].dropna()

fig2 = px.line(intensity, x="year", y="co2_per_gdp",
    title="📉 Global CO₂ Intensity (CO₂ per unit GDP) — Declining Since 1980s",
    labels={"co2_per_gdp":"CO₂ per unit GDP","year":"Year"},
    color_discrete_sequence=["#00c896"])
fig2.update_traces(line_width=2.5)
fig2.update_layout(template="plotly_dark", paper_bgcolor="#0b1a14",
    plot_bgcolor="#112a1e", height=380)
fig2.show()

print("\n💡 Key Insight: Richer nations emit more per person, but the world economy")
print("   is becoming more carbon-efficient (less CO₂ per $ of GDP) since the 1980s.")
print("   However, efficiency gains are not fast enough to offset economic growth.")

---
## ✅ Portfolio Summary

| # | Project | Key Finding |
|---|---------|------------|
| 1 | Global CO₂ Trend | Emissions rose 20× since 1900; COVID caused biggest-ever dip |
| 2 | Top Emitters | China & USA = 40% of global CO₂; Gulf states lead per capita |
| 3 | Temperature Impact | USA leads cumulative warming despite China emitting more today |
| 4 | Energy Sources | Coal still dominates; transition is uneven across regions |
| 5 | Emissions vs GDP | Richer = more emissions per person; intensity declining since 1980s |

---
**Tools used:** `pandas` · `plotly.express` · `plotly.graph_objects` · Jupyter Notebook  
**Data:** Our World in Data — CO₂ & GHG Emissions (CC BY 4.0)  
*Portfolio by Naveed Baig · Climate Data Analyst · 2025*